In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
print(torch.backends.cuda.is_flash_attention_available())

False


In [2]:
with open("data/tinyshakespeare/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

split = int(0.9 * len(text))

train_text = text[:split]
val_text = text[split:]

print(len(train_text), len(val_text))

1003854 111540


In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def decode(tokens):
    return ''.join(itos[i] for i in tokens)
print(vocab_size)


65


In [4]:
train_data = torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
val_data   = torch.tensor([stoi[c] for c in val_text], dtype=torch.long)

class ShakespeareDataset(Dataset):
    def __init__(self, data, seq_len) -> None:
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, index):
        x = self.data[index:index + self.seq_len] # the current letter
        y = self.data[index+ 1: index + self.seq_len + 1] # the next letter

        return x, y


seq_len = 256
batch_size = 64

train_dataset = ShakespeareDataset(train_data, seq_len)
val_dataset = ShakespeareDataset(val_data, seq_len)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
from torch import nn
from torch.nn import functional as F

batch_size = 64
block_size = seq_len

n_embed = 128
n_head = 4
n_layer = 4
n_kv_head = 2

dropout = 0.1
learning_rate = 3e-4

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd,n_head,n_kv_head,dropout=0.0) -> None:
        super().__init__()
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        

        self.q_proj = nn.Linear(n_embed,n_head * self.head_dim,bias=False)
        self.k_proj = nn.Linear(n_embed,n_kv_head * self.head_dim,bias=False)
        self.v_proj = nn.Linear(n_embed,n_kv_head * self.head_dim,bias=False)

        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout_p = dropout

    def forward(self,x,past_k = None,past_v = None, use_cache = False):
        B, T, C = x.shape

        #Q
        q =self.q_proj(x)
        q = q.view(B,T,self.n_head,self.head_dim)
        q = q.transpose(1,2)
        # (B, n_head, T, head_dim)

        # K
        k = self.k_proj(x)
        k = k.view(B,T,self.n_kv_head,self.head_dim)
        k = k.transpose(1, 2)
        # (B, n_kv_head, T, head_dim)

        #V
        v = self.v_proj(x)
        v = v.view(B,T,self.n_kv_head,self.head_dim)
        v = v.transpose(1, 2)
        # (B, n_kv_head, T, head_dim)

        #KV Cache
        if past_k is not None:
            k = torch.cat([past_k, k],dim=2)

        if past_v is not None:
            v = torch.cat([past_v, v],dim=2)

        #Att
        if past_k is None:
            # Training / prefill
            # Q length == K/V length
            # Therefore normal causal attention.

            out = F.scaled_dot_product_attention(q,k,v,
                is_causal=True,
                dropout_p=(self.dropout_p if self.training else 0.0),
                enable_gqa=True
            )

        else:
            # Decode with KV cache
            # q = only the NEW token
            # k/v = entire cache
            # The new token should attend to ALL
            # cached tokens, so DO NOT use is_causal=True.

            out = F.scaled_dot_product_attention(q,k,v,
                is_causal=False,
                dropout_p=0.0,
                enable_gqa=True
            )

        out = out.transpose(1, 2)

        out = out.contiguous().view(
            B,
            T,
            C
        )

        out = self.proj(out)

        if use_cache:
            return out, k, v

        return out


In [7]:
class FeedForward(nn.Module):
    def __init__(self,n_embed) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed,n_embed * 4),
            nn.GELU(),
            nn.Linear(n_embed * 4,n_embed),
            nn.Dropout(dropout)
        )

    def forward(self,x):
        return self.net(x)

In [8]:
class Block(nn.Module):
    def __init__(self,n_embed,n_head,n_kv_head) -> None:
        super().__init__()
        head_size = n_embed // n_head

        self.sa = MultiHeadAttention(
            n_embd=n_embed,
            n_head=n_head,
            n_kv_head=n_kv_head,
            dropout=dropout
        )

        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self,x,past_k = None, past_v = None, use_cache = False):
        x_norm = self.ln1(x)

        if use_cache:
            attn_out,k,v = self.sa(
                x_norm,
                past_k = past_k,
                past_v = past_v,
                use_cache = True
            )
        else:
            attn_out = self.sa(x_norm)

        x = x + attn_out

        x = x + self.ffwd(
            self.ln2(x)
        )

        if use_cache:
            return x, k, v

        return x

In [9]:
class Transformer(nn.Module):
    def __init__(self,n_embed,n_head,n_layer,n_kv_head) -> None:
        super().__init__()
        self.blocks = nn.ModuleList([
            Block(n_embed,n_head,n_kv_head) for _ in range(n_layer)
        ])
        self.token_embedding = nn.Embedding(vocab_size,n_embed)
        self.positional_embedding = nn.Embedding(block_size,n_embed)
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed,vocab_size)

    def forward(self,x, targets = None, past_kvs = None, use_cache = False):
        B,T = x.shape
        token_embed = self.token_embedding(x)

        #Positional Embed
        if past_kvs is None:
            positions = torch.arange(T, device=x.device)
        else:
            past_len = past_kvs[0][0].size(2)
            positions = torch.arange(past_len,past_len + T, device=x.device)

        position_embed = self.positional_embedding(positions)
        x = token_embed + position_embed

        # Transformer Blocks
        new_kvs = []

        for i, block in enumerate(self.blocks):
            if use_cache:
                past_k = None
                past_v = None

                if past_kvs is not None:
                    past_k, past_v = past_kvs[i]

                x,k,v = block(x, past_k = past_k, past_v = past_v, use_cache = True)
                new_kvs.append((k,v))
            else:
                x = block(x)
        
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B,T,C = logits.shape

            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B*T)

            loss = F.cross_entropy(logits_flat,targets_flat)
        if use_cache:
            return logits, loss, new_kvs

        return logits, loss

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Transformer(
    n_embed=n_embed,
    n_head=n_head,
    n_layer=n_layer,
    n_kv_head=n_kv_head
).to(device)


In [11]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)
scaler = torch.amp.GradScaler("cuda")

In [12]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_train_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, loss = model(xb, yb)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_train_loss += loss.item()

    avg_train_loss = (
        total_train_loss / len(train_loader)
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Train Loss: {avg_train_loss:.4f}"
    )

Epoch 1/10 | Train Loss: 1.5189
Epoch 2/10 | Train Loss: 1.2465
Epoch 3/10 | Train Loss: 1.1879
Epoch 4/10 | Train Loss: 1.1512
Epoch 5/10 | Train Loss: 1.1230
Epoch 6/10 | Train Loss: 1.0991
Epoch 7/10 | Train Loss: 1.0788
Epoch 8/10 | Train Loss: 1.0615
Epoch 9/10 | Train Loss: 1.0468
Epoch 10/10 | Train Loss: 1.0346


In [13]:
model.eval()

total_loss = 0
total_batches = 0

with torch.no_grad():

    for xb, yb in val_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, loss = model(xb, yb)

        total_loss += loss.item()
        total_batches += 1

val_loss = total_loss / total_batches
val_loss

1.6386260280943656

In [14]:
model.eval()

context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

past_kvs = None

with torch.no_grad():

    for step in range(256):

        if past_kvs is None:

            # First pass:
            # process the entire current context

            context_cond = context[:, -block_size:]

            logits, _, past_kvs = model(
                context_cond,
                use_cache=True
            )

        else:

            # Subsequent passes:
            # ONLY process the new token

            new_token = context[:, -1:]

            logits, _, past_kvs = model(
                new_token,
                past_kvs=past_kvs,
                use_cache=True
            )

        # Last token's logits

        logits = logits[:, -1, :]

        # Temperature can be added here
        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        context = torch.cat(
            (context, next_token),
            dim=1
        )

print(
    decode(
        context[0].tolist()
    )
)


That you are a waggon ship! you have made work!

RIVERS:
I may partly the king is kind of a grandam?

GLOUCESTER:
I'll k away. How for me towards with thee?

QUEEN MARGARET:
And that thou art well-amried!

KING HENRY VI:
Well, thou grandst not an execution
